# Experimento 3 · ESP32-CAM + TinyML + 2 HC-SR04 + LEDs integrados + HTML

- **Placa:** ESP32-CAM AI-Thinker con ESP32-CAM-MB.
- **Sensores (ya montados):** TRIG/ECHO izquierdo IO13/IO14 y derecho IO15/IO2, con divisores de tensión.
- **Indicadores:** LED izquierdo externo IO12 + rojo integrado IO33 (activo LOW); derecho externo IO4 + flash blanco integrado IO4.
- **Datos:** 8 000 registros **sintéticos**. El entrenamiento genera `model.h` con los **pesos reales aprendidos por Keras**, que el ESP32-CAM ejecuta localmente sin instalar `tflm_esp32` ni `EloquentTinyML`.

El cuaderno también exporta un `.tflite` como referencia, pero el firmware usa los pesos Dense float32 del archivo `model.h`. Este procedimiento sigue siendo inferencia TinyML en el microcontrolador. El panel recibe las predicciones a 115200 baudios por USB y no necesita Wi-Fi ni un puente de datos.


In [ ]:
import os, csv, random, collections, io, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix

SEED = 20261123
np.random.seed(42)
tf.keras.utils.set_random_seed(42)
CLASES = ['A', 'I', 'D', 'S']
NOMBRES = {'A':'AVANZAR', 'I':'IZQUIERDA', 'D':'DERECHA', 'S':'DETENER'}
UMBRAL_CM = 20.0
ESCALA_CM = 200.0
print('TensorFlow:', tf.__version__)


## 1. Generar 8 000 registros sintéticos (2 000 por decisión)

Se define **obstáculo cercano como distancia menor de 20 cm**; es el mismo umbral que se utiliza para los LEDs del ESP32-CAM. Se evitan valores muy próximos al umbral durante la generación inicial para reducir etiquetas ambiguas. Los datos son ficticios y NO sustituyen una prueba real.


In [ ]:
r = random.Random(SEED)

def sample_close():
    # Obstáculo presente: 4 a 19.3 cm
    v = r.random()
    if v < 0.30: x=r.uniform(4.0,9.0)
    elif v < 0.65: x=r.uniform(9.0,15.0)
    else: x=r.uniform(15.0,19.3)
    return round(x,2)

def sample_open():
    # Espacio libre: 20.7 a 190 cm
    v = r.random()
    if v < 0.35: x=r.uniform(20.7,40.0)
    elif v < 0.75: x=r.uniform(40.0,100.0)
    else: x=r.uniform(100.0,190.0)
    return round(x,2)

rows=[]
for clase in CLASES:
    for _ in range(2000):
        if clase=='A': izq,der=sample_open(),sample_open()
        elif clase=='I': izq,der=sample_open(),sample_close()
        elif clase=='D': izq,der=sample_close(),sample_open()
        else: izq,der=sample_close(),sample_close()
        rows.append((izq,der,clase))
r.shuffle(rows)

ruta_csv=Path('dataset_obstaculos_tinyml_8000.csv')
with ruta_csv.open('w',newline='',encoding='utf-8') as f:
    wr=csv.writer(f)
    wr.writerow(['izquierda_cm','derecha_cm','clase'])
    wr.writerows((f'{a:.2f}',f'{b:.2f}',c) for a,b,c in rows)

df=pd.read_csv(ruta_csv)
print('Registros:',len(df))
print(df['clase'].value_counts().sort_index())
display(df.head(10))
assert len(df)==8000
assert set(df['clase'])==set(CLASES)


## 2. Análisis de datos (EDA)

Los pares de sensores se distribuyen en cuatro regiones, una para cada decisión. Esta separación procede de las reglas con las que **generamos** el dataset.

In [ ]:
fig, ax=plt.subplots(figsize=(8,6))
for cl in CLASES:
    sub=df[df.clase==cl]
    ax.scatter(sub.izquierda_cm,sub.derecha_cm,s=8,alpha=.35,label=NOMBRES[cl])
ax.axvline(UMBRAL_CM,linestyle='--',linewidth=1)
ax.axhline(UMBRAL_CM,linestyle='--',linewidth=1)
ax.set_xlabel('Distancia izquierda (cm)')
ax.set_ylabel('Distancia derecha (cm)')
ax.set_title('Datos sintéticos por decisión')
ax.legend()
plt.show()


## 3. Dividir datos y entrenar la red neuronal

Dos entradas normalizadas por 200 cm; dos capas ocultas `Dense(24)` y `Dense(16)`; una salida `Softmax(4)`. Separamos entrenamiento, validación y prueba sin mezclar sus etiquetas.

In [ ]:
X=df[['izquierda_cm','derecha_cm']].to_numpy(dtype=np.float32)
y=np.array([CLASES.index(c) for c in df['clase']],dtype=np.int32)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,y,test_size=0.20,random_state=42,stratify=y
)
X_train, X_val, y_train, y_val=train_test_split(
    X_trainval,y_trainval,test_size=0.20,random_state=42,stratify=y_trainval
)

def normalizar(X):
    return np.clip(np.asarray(X,dtype=np.float32),0.0,ESCALA_CM)/ESCALA_CM

X_train=normalizar(X_train)
X_val=normalizar(X_val)
X_test=normalizar(X_test)
print('Entrenamiento:',len(X_train),'Validación:',len(X_val),'Prueba:',len(X_test))

red=tf.keras.Sequential([
    tf.keras.Input(shape=(2,),dtype=tf.float32),
    tf.keras.layers.Dense(24,activation='relu'),
    tf.keras.layers.Dense(16,activation='relu'),
    tf.keras.layers.Dense(4,activation='softmax')
])
red.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
            loss='sparse_categorical_crossentropy',metrics=['accuracy'])
red.summary()

hist=red.fit(
    X_train,y_train,validation_data=(X_val,y_val),
    batch_size=64,epochs=160,verbose=1,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',patience=18,restore_best_weights=True
    )]
)


## 4. Evaluación del modelo (solo sobre datos sintéticos no vistos)

Estos resultados **no prueban** la precisión del dispositivo en un entorno real; se necesitarán mediciones nuevas con los sensores para esa conclusión.

In [ ]:
loss,accuracy=red.evaluate(X_test,y_test,verbose=0)
y_pred=np.argmax(red.predict(X_test,verbose=0),axis=1)
print('Precisión en PRUEBA SINTÉTICA: %.2f %%'%(accuracy*100))
print(classification_report(y_test,y_pred,target_names=[NOMBRES[c] for c in CLASES],digits=4))

fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(hist.history['loss'],label='Entrenamiento')
axes[0].plot(hist.history['val_loss'],label='Validación')
axes[0].set_title('Pérdida')
axes[0].legend()
axes[1].plot(hist.history['accuracy'],label='Entrenamiento')
axes[1].plot(hist.history['val_accuracy'],label='Validación')
axes[1].set_title('Precisión')
axes[1].legend()
plt.show()

cm=confusion_matrix(y_test,y_pred,labels=list(range(4)))
ConfusionMatrixDisplay(cm,display_labels=[NOMBRES[c] for c in CLASES]).plot(cmap='Blues',xticks_rotation=25)
plt.title('Matriz de confusión — prueba sintética')
plt.show()


## 5. Exportar TFLite y `model.h` compatible con Arduino (sin bibliotecas TFLM)

Exportaremos las matrices y sesgos de las capas Dense de Keras como arreglos `float` C++, en el mismo orden en que los usa el sketch. Para asegurar que la inferencia en el ESP32-CAM reproduce la red, comprobaremos matemáticamente los resultados en Python antes de exportar.


In [ ]:
# Archivo TFLite opcional para comparar/exportar el modelo aprendido.
converter = tf.lite.TFLiteConverter.from_keras_model(red)
tflite_bytes = converter.convert()
Path('modelo_obstaculos.tflite').write_bytes(tflite_bytes)
print('Modelo TFLite para referencia:', len(tflite_bytes), 'bytes')

# Extraer EXACTAMENTE los pesos entrenados de las 3 capas Dense.
capas = [c for c in red.layers if isinstance(c, tf.keras.layers.Dense)]
assert len(capas) == 3
assert [int(c.units) for c in capas] == [24, 16, 4]
assert [c.activation.__name__ for c in capas] == ['relu', 'relu', 'softmax']

def numero_float_cpp(x):
    s = format(float(np.float32(x)), '.9g')
    if '.' not in s and 'e' not in s:
        s += '.0'
    return s + 'f'

def declaracion_cpp(nombre, a):
    valores = np.asarray(a, dtype=np.float32).reshape(-1)
    lineas = [f'static const float {nombre}[{len(valores)}] = {{']
    for inicio in range(0, len(valores), 8):
        trozo = valores[inicio:inicio+8]
        lineas.append('  ' + ', '.join(numero_float_cpp(x) for x in trozo) + (',' if inicio + 8 < len(valores) else ''))
    lineas.append('};')
    return '\n'.join(lineas)

cabecera = [
    '#pragma once',
    '#define EXP3_MODELO_PESOS_V1 1',
    '// Entrenado en Google Colab con Keras: red 2 -> 24 -> 16 -> 4',
    '// Pesos float32 Dense en orden [entrada,salida], seguidos de sesgos.',
    '// Entradas normalizadas: clip(distancia_cm,0,200)/200',
    '// Salida: 0=AVANZAR, 1=IZQUIERDA, 2=DERECHA, 3=DETENER',
]
for indice, capa in enumerate(capas, start=1):
    pesos, sesgos = capa.get_weights()
    cabecera.append(declaracion_cpp(f'PESOS_{indice}', pesos))
    cabecera.append(declaracion_cpp(f'SESGOS_{indice}', sesgos))
Path('model.h').write_text('\n\n'.join(cabecera) + '\n', encoding='utf-8')
for nombre in ['PESOS_1', 'SESGOS_1', 'PESOS_2', 'SESGOS_2', 'PESOS_3', 'SESGOS_3']:
    assert nombre in Path('model.h').read_text(encoding='utf-8'), f'Falta {nombre}'
print('Archivo model.h creado:', Path('model.h').stat().st_size, 'bytes')

# La inferencia propia (matrices, ReLU y Softmax) coincide con Keras.
xs = normalizar([[80, 80], [80, 10], [10, 80], [10, 10], [16.16, 17.24], [22, 22]])
def prediccion_sin_tensorflow(xs):
    z = xs.copy()
    for idx, capa in enumerate(capas):
        pesos, sesgos = capa.get_weights()
        z = z @ pesos + sesgos
        if idx < 2:
            z = np.maximum(z, 0)
        else:
            exp = np.exp(z - z.max(axis=1, keepdims=True))
            z = exp / exp.sum(axis=1, keepdims=True)
    return z
p_py = prediccion_sin_tensorflow(xs)
p_tf = red.predict(xs, verbose=0)
np.testing.assert_allclose(p_py, p_tf, rtol=0.0001, atol=0.0001)
for distancias, prob in zip(xs * 200, p_py):
    k = int(np.argmax(prob))
    print('Izquierda %.2f cm | derecha %.2f cm -> %s %.1f%%' %
          (distancias[0], distancias[1], NOMBRES[CLASES[k]], 100 * prob[k]))


## 6. Descargar el modelo para Arduino IDE

`Modelo_TinyML_Para_Arduino.zip` contiene `model.h` (matrices y sesgos), `.tflite` (referencia), CSV y métricas. Puedes sustituir el `model.h` preentrenado que ya acompaña al proyecto por este generado en Colab, en la carpeta `Arduino/02_TinyML_Sensores_LEDs/`.

**NO instales `tflm_esp32` ni `EloquentTinyML`:** el sketch actualizado ejecuta los pesos localmente con operaciones numéricas de C++ y evita el error de símbolos duplicados de esas bibliotecas. El panel HTML usa Web Serial del navegador; ejecútalo mediante `Panel_HTML/abrir_local.cmd`, que abre un servidor **estático** local sin puente USB.


In [ ]:
reporte=(f'Experimento 3 - ESP32-CAM / TinyML / LEDs / HTML local\n'
         f'Filas sintéticas: {len(df)}\n'
         f'Entrenamiento: {len(X_train)} | Validación: {len(X_val)} | Prueba: {len(X_test)}\n'
         f'Precisión en prueba SINTÉTICA: {accuracy*100:.2f}%\n'
         f'Orden: 0=AVANZAR, 1=IZQUIERDA, 2=DERECHA, 3=DETENER\n'
         f'Normalización: clip(cm, 0, 200)/200\n'
         f'Modelo: red neuronal Dense 2-24-16-4, pesos exportados a model.h\n'
         'CUIDADO: prueba SINTÉTICA, NO demuestra fiabilidad real.\n')
Path('resultados_entrenamiento.txt').write_text(reporte,encoding='utf-8')
with zipfile.ZipFile('Modelo_TinyML_Para_Arduino.zip','w',compression=zipfile.ZIP_DEFLATED) as z:
    for nombre in ['model.h','modelo_obstaculos.tflite','dataset_obstaculos_tinyml_8000.csv','resultados_entrenamiento.txt']:
        z.write(nombre)
print(reporte)
try:
    from google.colab import files
    files.download('Modelo_TinyML_Para_Arduino.zip')
except ImportError:
    print('Descarga manual:',os.path.abspath('Modelo_TinyML_Para_Arduino.zip'))
